In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../tables/part_1")

consumer_path = DATA_DIR / "tbl_consumer.csv"
consumer_details_path = DATA_DIR / "consumer_user_details.parquet"
merchant_path = DATA_DIR / "tbl_merchants.parquet"

In [3]:
consumer = pd.read_csv(
    consumer_path,
    sep="|"
)

consumer_details = pd.read_parquet(
    consumer_details_path
)

merchants = pd.read_parquet(
    merchant_path
)

In [4]:
def clean_consumer(df):
    df = df.copy()

    # Postcodes are identifiers rather than numeric measurements.
    # Restore leading zeroes lost during CSV parsing.
    df["postcode"] = (
        df["postcode"]
        .astype(str)
        .str.zfill(4)
    )

    return df

In [5]:
def validate_merchant_fraud(df):
    # Required fields must be present
    assert df["merchant_abn"].notna().all(), \
        "Missing merchant ABN"

    assert df["order_datetime"].notna().all(), \
        "Missing order datetime"

    assert df["fraud_probability"].notna().all(), \
        "Missing fraud probability"

    # Fraud probability must be on the observed 0-100 scale
    assert df["fraud_probability"].between(0, 100).all(), \
        "Fraud probability outside 0-100 range"

    # Dates should have been converted during cleaning
    assert pd.api.types.is_datetime64_any_dtype(df["order_datetime"]), \
        "order_datetime is not datetime"

In [6]:
# validation functions 
VALID_STATES = {
    "ACT", "NSW", "NT", "QLD",
    "SA", "TAS", "VIC", "WA"
}

VALID_GENDERS = {
    "Male", "Female", "Undisclosed"
}


def validate_consumer(df):
    assert df["consumer_id"].notna().all(), \
        "Missing consumer IDs detected"

    assert df["consumer_id"].is_unique, \
        "Duplicate consumer IDs detected"

    assert df["state"].isin(VALID_STATES).all(), \
        "Invalid state detected"

    assert df["gender"].isin(VALID_GENDERS).all(), \
        "Invalid gender detected"

    assert df["postcode"].str.len().eq(4).all(), \
        "Invalid postcode length detected"

    assert df["name"].str.strip().ne("").all(), \
        "Blank consumer name detected"

    assert df["address"].str.strip().ne("").all(), \
        "Blank consumer address detected"

In [7]:
#consumer details 
def validate_consumer_details(df):
    assert df["consumer_id"].notna().all()
    assert df["user_id"].notna().all()

    assert df["consumer_id"].is_unique, \
        "Duplicate consumer IDs in mapping table"

    assert df["user_id"].is_unique, \
        "Duplicate user IDs in mapping table"
    
# merchant details
def validate_merchants(df):
    assert df.index.notna().all(), \
        "Missing merchant ABN"

    assert df.index.is_unique, \
        "Duplicate merchant ABN"

    assert (
        pd.Series(df.index.astype(str))
        .str.len()
        .eq(11)
        .all()
    ), "Invalid merchant ABN length"

    assert df["name"].notna().all(), \
        "Missing merchant names"

    assert df["name"].str.strip().ne("").all(), \
        "Blank merchant names"
    

# relationship validation 
def validate_consumer_relationship(consumer, consumer_details):

    consumer.merge(
        consumer_details,
        on="consumer_id",
        validate="one_to_one"
    )

    assert set(consumer["consumer_id"]) == \
           set(consumer_details["consumer_id"]), \
           "Consumer IDs do not match between tables"

In [8]:
# loading functions 
def load_consumer(path):
    return pd.read_csv(path, sep="|")


def load_consumer_details(path):
    return pd.read_parquet(path)


def load_merchants(path):
    return pd.read_parquet(path)


def load_consumer_fraud(path):
    return pd.read_csv(path)


def load_merchant_fraud(path):
    return pd.read_csv(path)

In [9]:
def clean_consumer_fraud(df):
    df = df.copy()

    # Remove exact duplicate fraud observations
    df = df.drop_duplicates()

    # Convert dates to datetime
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"],
        errors="raise"
    )

    return df

In [10]:
def clean_merchant_fraud(df):
    df = df.copy()

    # Convert dates to datetime
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"],
        errors="raise"
    )

    return df

In [11]:
def validate_consumer_fraud(df):
    assert df["user_id"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(0, 100).all()

    assert df.duplicated().sum() == 0

In [12]:
consumer_clean = clean_consumer(consumer)
consumer_clean["postcode"].str.len().value_counts()

postcode
4    499999
Name: count, dtype: int64

In [13]:
#final one 
def run_part1_pipeline():

    # --------------------
    # LOAD
    # --------------------

    consumer = pd.read_csv(
        DATA_DIR / "tbl_consumer.csv",
        sep="|"
    )

    consumer_details = pd.read_parquet(
        DATA_DIR / "consumer_user_details.parquet"
    )

    merchants = pd.read_parquet(
        DATA_DIR / "tbl_merchants.parquet"
    )

    consumer_fraud = pd.read_csv(
        DATA_DIR / "consumer_fraud_probability.csv"
    )

    merchant_fraud = pd.read_csv(
        DATA_DIR / "merchant_fraud_probability.csv"
    )


    # --------------------
    # CLEAN
    # --------------------

    consumer = clean_consumer(consumer)

    consumer_fraud = clean_consumer_fraud(
        consumer_fraud
    )

    merchant_fraud = clean_merchant_fraud(
        merchant_fraud
    )


    # --------------------
    # VALIDATE
    # --------------------

    validate_consumer(consumer)

    validate_consumer_details(
        consumer_details
    )

    validate_merchants(
        merchants
    )

    validate_consumer_fraud(
        consumer_fraud
    )

    validate_merchant_fraud(
        merchant_fraud
    )

    validate_consumer_relationship(
        consumer,
        consumer_details
    )


    # --------------------
    # RETURN CLEAN TABLES
    # --------------------

    return (
        consumer,
        consumer_details,
        merchants,
        consumer_fraud,
        merchant_fraud
    )

In [14]:
(
    consumer_clean,
    consumer_details_clean,
    merchants_clean,
    consumer_fraud_clean,
    merchant_fraud_clean
) = run_part1_pipeline()

In [15]:
consumer_enriched = consumer_clean.merge(
    consumer_details_clean,
    on="consumer_id",
    how="left",
    validate="one_to_one"
)

In [16]:
print(consumer_enriched.shape)
print(consumer_enriched["user_id"].isna().sum())

(499999, 7)
0


In [17]:
"""
consumer                  -> cleaned + validated
consumer_user_details     -> validated
merchants                 -> validated
consumer_fraud            -> cleaned + validated
merchant_fraud            -> cleaned + validated
 
 and created a merged "consumer enriched" table with consumer and consumer_user_details
 

"""

'\nconsumer                  -> cleaned + validated\nconsumer_user_details     -> validated\nmerchants                 -> validated\nconsumer_fraud            -> cleaned + validated\nmerchant_fraud            -> cleaned + validated\n \n and created a merged "consumer enriched" table with consumer and consumer_user_details\n \n\n'

In [18]:
# bringing in the cleaning functions for part 2,3,4 

In [19]:
TABLES_DIR = Path.cwd() / "../tables"


def read_partitioned_parquet(path):
    path = Path(path)

    if not path.exists():
        return None

    try:
        return (
            spark.read
            .option("basePath", str(path))
            .parquet(str(path))
        )
    except Exception as exc:
        print(f"Could not load partitioned parquet dataset from {path}: {exc}")
        return None
    


In [20]:
def load_all_transactions(tables_dir):
    transaction_frames = []

    for part_name in ["part_2", "part_3", "part_4"]:
        part_path = TABLES_DIR / part_name
        df = read_partitioned_parquet(part_path)

        if df is not None:
            transaction_frames.append(df)

    if not transaction_frames:
        raise ValueError("No transaction datasets could be loaded.")

    transactions = transaction_frames[0]

    for df in transaction_frames[1:]:
        transactions = transactions.unionByName(
            df,
            allowMissingColumns=True
        )

    return transactions

In [21]:
from pathlib import Path
import pandas as pd

TABLES_DIR = Path.cwd().parent / "tables"

def load_all_transactions(tables_dir):
    frames = []

    for part in ["part_2", "part_3", "part_4"]:
        path = Path(tables_dir) / part

        if not path.exists():
            raise FileNotFoundError(f"Missing: {path}")

        print(f"Loading {part}...")
        df = pd.read_parquet(path)
        frames.append(df)

    transactions = pd.concat(frames, ignore_index=True)

    return transactions

In [22]:
transactions = load_all_transactions(TABLES_DIR) 

Loading part_2...
Loading part_3...
Loading part_4...


In [23]:
# joining function 
def join_consumer_data(transactions, consumer_enriched):
    return transactions.merge(
        consumer_enriched,
        on="user_id",
        how="left",
        validate="many_to_one"
    )



In [24]:
merchants_clean = merchants_clean.reset_index()
print(merchants_clean.columns.tolist())

['merchant_abn', 'name', 'tags']


In [25]:
def join_merchant_data(transactions, merchants_clean):
    return transactions.merge(
        merchants_clean,
        on="merchant_abn",
        how="left",
        validate="many_to_one"
    )

In [26]:
print(type(transactions))

<class 'pandas.core.frame.DataFrame'>


In [27]:
curated_transactions = join_consumer_data(
    transactions,
    consumer_enriched
)

curated_transactions = join_merchant_data(
    curated_transactions,
    merchants_clean
)

In [29]:
# Dimensions
print("Shape:", curated_transactions.shape)

# Fields + data types
print("\nFIELDS:")
print(curated_transactions.dtypes)



Shape: (14195505, 13)

FIELDS:
user_id             int64
merchant_abn        int64
dollar_value      float64
order_id           object
order_datetime     object
name_x             object
address            object
state              object
postcode           object
gender             object
consumer_id         int64
name_y             object
tags               object
dtype: object


In [30]:
print(curated_transactions.isna().sum())

user_id                0
merchant_abn           0
dollar_value           0
order_id               0
order_datetime         0
name_x                 0
address                0
state                  0
postcode               0
gender                 0
consumer_id            0
name_y            580830
tags              580830
dtype: int64


In [ ]:
# concern, we have some unmatched merchant rows. This is expected, as the merchant fraud probability table is a sample of merchants, not the full set.
print("Total transactions:", len(curated_transactions))
print("Unmatched merchant rows:", curated_transactions["name_y"].isna().sum())

print(
    "Unmatched %:",
    curated_transactions["name_y"].isna().mean() * 100
)

Total transactions: 14195505
Unmatched merchant rows: 580830
Unmatched %: 4.091647320753999
